# Adverse selection and markouts

Thin wrapper over `ratesresearch.markouts` on the two fitting-period sessions committed under `data/fixtures/backtest/fitting/` (fixed-spread strategy, central band point). The informed-share estimate is compared with the generator's truth file, which only a synthetic world has.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import SVG, Markdown, display

from ratesresearch import backtest_analysis as ba
from ratesresearch import presets

ROOT = Path.cwd()
while not (ROOT / "data" / "presets.yaml").is_file():
    ROOT = ROOT.parent
PRESETS = ROOT / "data" / "presets.yaml"


def show(df: pd.DataFrame) -> None:
    display(Markdown(ba.to_markdown(df)))

In [ ]:
from ratesresearch import markouts

days = sorted((ROOT / "data/fixtures/backtest/fitting").iterdir())
fills = pd.concat([pd.read_csv(d / "fills.csv") for d in days], ignore_index=True)
rfqs = pd.concat([pd.read_csv(d / "rfqs.csv") for d in days], ignore_index=True)
truth = pd.concat([pd.read_csv(d / "truth.csv") for d in days], ignore_index=True)
show(markouts.curves(fills, "tier").round(4))
show(markouts.curves(fills, "size_bucket").round(4))
show(markouts.curves(fills, "instrument").round(4))

In [ ]:
est = markouts.informed_share(fills, "1min", by="tier")
cf = fills[fills["kind"] == "client"].merge(truth, left_on="ref_id", right_on="rfq_id")
est["truth"] = est["group"].map(cf.groupby("tier")["informed"].mean())
show(est.round(4))
show(markouts.toxicity_table(fills, rfqs).round(4))